# 1.3 Regresión polinómica y regularización


**Objetivo:** Comparar regresión lineal, polinómica, Ridge y Lasso en un mismo problema.  
**Dataset:** `rendimiento_estudiantil.csv`



## Sesión

1. Cargar y revisar el dataset.  
2. Preparar variables de entrada y salida.  
3. Entrenar el modelo.  
4. Evaluar resultados.  
5. Interpretar lo que ocurrió.  
6. Resolver una actividad.


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving rendimiento_estudiantil.csv to rendimiento_estudiantil (1).csv


In [ ]:
import pandas as pd

df = pd.read_csv('rendimiento_estudiantil.csv')
df.head()

,horas_estudio,asistencia_pct,tareas_entregadas,participacion,horas_sueno,promedio_final
0,9.9,79.0,10,4.8,6.3,100.0
1,5.9,85.2,7,6.3,6.2,96.7
2,11.3,71.9,9,6.1,6.8,100.0
3,11.8,74.3,9,8.4,8.5,100.0
4,3.1,100.0,7,7.4,8.9,94.0


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
from pathlib import Path
import pandas as pd

def cargar_csv(nombre_archivo):
    rutas = [
        Path(nombre_archivo),
        Path("../datasets") / nombre_archivo,
        Path("datasets") / nombre_archivo,
        Path("/content") / nombre_archivo,
        Path("/content/datasets") / nombre_archivo,
    ]
    for ruta in rutas:
        if ruta.exists():
            print(f"Archivo encontrado en: {ruta}")
            return pd.read_csv(ruta)
    raise FileNotFoundError(
        f"No se encontró {nombre_archivo}. Súbelo a Colab o colócalo en la carpeta datasets."
    )


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score


In [ ]:
cldf = cargar_csv("rendimiento_estudiantil.csv")
X = df.drop(columns="promedio_final")
y = df["promedio_final"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

modelos = {
    "Lineal": Pipeline([
        ("reg", LinearRegression())
    ]),
    "Polinómico grado 2": Pipeline([
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("reg", LinearRegression())
    ]),
    "Ridge": Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=1.0))
    ]),
    "Lasso": Pipeline([
        ("scaler", StandardScaler()),
        ("lasso", Lasso(alpha=0.2))
    ])
}

resultados = []
for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)
    resultados.append({
        "modelo": nombre,
        "RMSE_test": np.sqrt(mean_squared_error(y_test, pred)),
        "R2_test": r2_score(y_test, pred)
    })

pd.DataFrame(resultados).round(4).sort_values("RMSE_test")


Archivo encontrado en: rendimiento_estudiantil.csv


,modelo,RMSE_test,R2_test
1,Polinómico grado 2,2.9747,0.4058
3,Lasso,3.2952,0.2708
2,Ridge,3.3289,0.2559
0,Lineal,3.3301,0.2553


## ¿Qué estamos comparando?

- **Lineal:** usa una relación directa entre entradas y salida.
- **Polinómico:** crea combinaciones y curvaturas.
- **Ridge:** penaliza coeficientes grandes y ayuda a estabilizar.
- **Lasso:** además puede llevar algunos coeficientes a cero.


In [ ]:
modelo_ridge = modelos["Ridge"]
modelo_ridge.fit(X_train, y_train)

coeficientes_ridge = pd.DataFrame({
    "variable": X.columns,
    "coeficiente_ridge": np.round(modelo_ridge.named_steps["ridge"].coef_, 4)
})

coeficientes_ridge


,variable,coeficiente_ridge
0,horas_estudio,1.9476
1,asistencia_pct,0.4722
2,tareas_entregadas,1.0361
3,participacion,0.8876
4,horas_sueno,-0.2624


In [ ]:
modelo_lasso = modelos["Lasso"]
modelo_lasso.fit(X_train, y_train)

coeficientes_lasso = pd.DataFrame({
    "variable": X.columns,
    "coeficiente_lasso": np.round(modelo_lasso.named_steps["lasso"].coef_, 4)
})

coeficientes_lasso


,variable,coeficiente_lasso
0,horas_estudio,1.7135
1,asistencia_pct,0.2478
2,tareas_entregadas,0.7920
3,participacion,0.6663
4,horas_sueno,-0.0623


## Actividad

1. Cambia `alpha` de Ridge a `0.1`, `1.0` y `10.0`.  
2. Cambia `alpha` de Lasso a `0.05`, `0.2` y `0.8`.  
3. Observa qué ocurre con las métricas y con los coeficientes.


In [ ]:
# Espacio para experimentar con distintos valores de alpha

df = cargar_csv("rendimiento_estudiantil.csv")
X = df.drop(columns="promedio_final")
y = df["promedio_final"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

modelos = {
    "Lineal": Pipeline([
        ("reg", LinearRegression())
    ]),
    "Polinómico grado 2": Pipeline([
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("reg", LinearRegression())
    ]),
    "Ridge": Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=0.1))
    ]),
    "Lasso": Pipeline([
        ("scaler", StandardScaler()),
        ("lasso", Lasso(alpha=0.05))
    ])
}

resultados = []
for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)
    resultados.append({
        "modelo": nombre,
        "RMSE_test": np.sqrt(mean_squared_error(y_test, pred)),
        "R2_test": r2_score(y_test, pred)
    })

pd.DataFrame(resultados).round(4).sort_values("RMSE_test")


Archivo encontrado en: rendimiento_estudiantil.csv


,modelo,RMSE_test,R2_test
1,Polinómico grado 2,2.9747,0.4058
3,Lasso,3.3151,0.2620
2,Ridge,3.3300,0.2554
0,Lineal,3.3301,0.2553


## Conclusión

Escribe una respuesta corta:
¿cuándo convendría usar regularización en vez de una regresión lineal simple?

Conviene usar regularización en vez de una regresión lineal simple cuando hay muchas variables o variables muy parecidas entre sí, porque ayuda a evitar el sobreajuste y mejora la predicción en datos nuevos.
